# End-to-End MLP Workflow

This notebook is a fully worked example of how to use `portfolio_toolkit` with a basic MLP-style forecasting model.

It covers the full notebook-first workflow:

1. load a shared dataset
2. add built-in toolkit features
3. engineer new custom notebook-local features
4. build forward return / alpha / volatility targets
5. split into train / validation / test using the shared split rules
6. define and train a small PyTorch MLP regressor
7. emit a standardized prediction table
8. turn predictions into a `PortfolioWeights` object
9. run the shared backtest
10. write reports and artifacts
11. log everything to MLflow

This is intentionally simple and heavily commented. The idea is that a teammate can copy this notebook, replace the model body, keep the shared data/evaluation layer, and still be comparable to everyone else.

## Running This Notebook In Colab

If you want to run this notebook in Google Colab, start by cloning the repo into the Colab session and installing the toolkit in editable mode.

Steps:

1. Set `REPO_URL` below to your GitHub repo URL.
2. Run the bootstrap cell once.
3. After that, the rest of the notebook can import `portfolio_toolkit` normally.

If you are running locally, the same cell will automatically fall back to the repository on your machine.


In [15]:
# Colab / local bootstrap cell
# - In Colab: clone the repo, install the package, and point repo_root at /content/...
# - Locally: point repo_root at this repository on disk and add src/ to sys.path

import os
import sys
from pathlib import Path

def is_repo_root(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'portfolio_toolkit').exists()

def local_repo_candidates() -> list[Path]:
    candidates = []

    if 'repo_root' in globals():
        candidates.append(Path(repo_root).expanduser())

    pwd = os.environ.get('PWD')
    if pwd:
        candidates.append(Path(pwd).expanduser())

    try:
        candidates.append(Path.cwd())
    except FileNotFoundError:
        pass

    expanded = []
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except FileNotFoundError:
            continue
        expanded.append(resolved)
        expanded.extend(resolved.parents)

    seen = set()
    ordered = []
    for candidate in expanded:
        candidate_str = str(candidate)
        if candidate_str not in seen:
            seen.add(candidate_str)
            ordered.append(candidate)

    return ordered

local_repo_root = next((path for path in local_repo_candidates() if is_repo_root(path)), None)
IN_COLAB = local_repo_root is None and 'google.colab' in sys.modules and Path('/content').exists()

if local_repo_root is not None:
    repo_root = local_repo_root
    os.chdir(repo_root)
elif IN_COLAB:
    REPO_URL = 'https://github.com/adamthorne27/Portfolio-Optimization-Lib.git'  # replace with your real repo URL
    REPO_DIR = '/content/Portfolio-Optimizer'

    if '<your-user-or-org>' in REPO_URL or '<your-repo>' in REPO_URL:
        raise ValueError('Set REPO_URL to your real GitHub repository URL before running this cell in Colab.')

    os.chdir('/')
    !rm -rf "$REPO_DIR"
    !git clone "$REPO_URL" "$REPO_DIR"
    %cd "$REPO_DIR"
    %pip install -e ".[dev]"

    repo_root = Path(REPO_DIR).resolve()
else:
    raise RuntimeError(
        'Could not locate the repository root automatically. Set repo_root to the repo path and rerun this cell.'
    )

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

try:
    current_workdir = Path.cwd()
except FileNotFoundError:
    current_workdir = Path(os.environ.get('PWD', '/'))

print('repo_root =', repo_root)
print('python =', sys.executable)
print('cwd =', current_workdir)
print('src_path =', src_path)


repo_root = /content/Portfolio-Optimizer
python = /usr/bin/python3
cwd = /content/Portfolio-Optimizer
src_path = /content/Portfolio-Optimizer/src


## What This Example Is Doing

This notebook uses `shared_set_2` by default instead of `shared_set_1`.

Why:

- `shared_set_1` is the full S&P 500 universe, so the first download is much larger.
- `shared_set_2` is smaller and faster for a first MLP example.

Once you understand the pattern, you can switch `dataset_name` to `shared_set_1` and run the exact same workflow over a much broader universe.

In [14]:
editable_target = f"{repo_root}[dev]"
%pip install -e "{editable_target}"

Obtaining file:///content/Portfolio-Optimizer
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for portfolio-toolkit (pyproject.toml) ... done
  Created wheel for portfolio-toolkit: filename=portfolio_toolkit-0.1.0-0.editable-py3-none-any.whl size=2690 sha256=8e6604fb6204ef1d70281c99240129ec10538127d4c7691bfbe9c28c1b2e8221
  Stored in directory: /tmp/pip-ephem-wheel-cache-oschhj6l/wheels/3c/af/c8/784fd6245b8910667dbe5211f4495a7a97e262f493a467aabd
Successfully built portfolio-toolkit
  Attempting uninstall: portfolio-toolkit
    Found existing installation: portfolio-toolkit 0.1.0
    Uninstalling portfolio-toolkit-0.1.0:
      Successfully uninstalled portfolio-toolkit-0.1.0


In [16]:
from pathlib import Path
import numpy as np
import pandas as pd

from portfolio_toolkit import (
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    make_forward_realized_vol_target,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_risk_adjusted,
    backtest_weights,
    write_backtest_artifacts,
)

# ---------------------------------------------------------------------
# Basic run configuration.
# ---------------------------------------------------------------------
# repo_root:
#   Where the toolkit config files, cache, MLflow folder, and runs folder live.
# dataset_name:
#   Which shared ticker universe + split rules we want to use.
# horizon:
#   Our prediction horizon in trading days.
# output_dir:
#   Where this notebook will write artifacts like metrics and QuantStats.
# ---------------------------------------------------------------------

repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()
dataset_name = 'shared_set_2'
model_name = 'torch_mlp_forecast_example'
horizon = 5
output_dir = repo_root / 'runs' / 'mlp_end_to_end_workflow'
output_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(dataset_name, repo_root=repo_root)
splits = split_dates(dataset_name, repo_root=repo_root)

print('Dataset preset:', dataset_name)
print('Dataset display name:', spec.name)
print('Number of modeled tickers:', len(spec.tickers))
print('Benchmark ticker:', spec.benchmark_ticker)
print('Train/Val/Test windows:', splits)

Dataset preset: shared_set_2
Dataset display name: growth_tech_innovation
Number of modeled tickers: 26
Benchmark ticker: SPY
Train/Val/Test windows: {'train': (Timestamp('2014-01-02 00:00:00'), Timestamp('2019-12-31 00:00:00')), 'val': (Timestamp('2020-01-02 00:00:00'), Timestamp('2021-12-31 00:00:00')), 'test': (Timestamp('2022-01-03 00:00:00'), Timestamp('2025-12-31 00:00:00'))}


## 1. Load Shared Price Data

The toolkit's `load_prices(...)` function is the shared data entrypoint.

What it does:

- reads the selected dataset preset from `configs/datasets.toml`
- downloads daily OHLCV data with `yfinance` if it is not already cached
- always includes `SPY` as the benchmark series
- validates and normalizes the dataframe before returning it

This is one of the main standardization points in the repo: everyone starts from the same dataset preset and the same split boundaries.

In [17]:
prices = load_prices(dataset_name, repo_root=repo_root)

print('Price frame shape:', prices.shape)
print('Date range:', prices['date'].min(), '->', prices['date'].max())
print('Number of unique tickers in price frame:', prices['ticker'].nunique())
display(prices.head())

Price frame shape: (78468, 8)
Date range: 2014-01-02 00:00:00 -> 2025-12-31 00:00:00
Number of unique tickers in price frame: 26


,date,ticker,open,high,low,close,adj_close,volume
0,2014-01-02,AAPL,19.845715,19.893929,19.715000,19.754642,17.140663,234684800
1,2014-01-03,AAPL,19.745001,19.775000,19.301071,19.320715,16.764154,392467600
2,2014-01-06,AAPL,19.194643,19.528570,19.057142,19.426071,16.855566,412610800
3,2014-01-07,AAPL,19.440001,19.498571,19.211430,19.287144,16.735029,317209200
4,2014-01-08,AAPL,19.243214,19.484285,19.238930,19.409286,16.841002,258529600


## 2. Add Built-In Toolkit Features

We start with a moderate built-in feature set.

These are all created by the shared library, which means other teammates can use the same starting point if they want:

- momentum
- volatility
- RSI
- moving-average distance
- volume z-score
- benchmark-relative return
- intraday range and beta vs SPY

This is a good example of the intended workflow:

- shared features for consistency and speed
- custom notebook-local features on top when you want to experiment

In [18]:
base_feature_names = [
    'return_5d',
    'return_20d',
    'vol_20d',
    'momentum_20d',
    'momentum_60d',
    'rsi_14',
    'price_to_sma_20d',
    'price_to_sma_50d',
    'volume_zscore_20d',
    'excess_return_20d_vs_spy',
    'intraday_range',
    'beta_20d_spy',
]

base_features = build_features(prices, feature_names=base_feature_names)
print('Base feature frame shape:', base_features.shape)
display(base_features)

Base feature frame shape: (78468, 14)


,date,ticker,return_5d,return_20d,vol_20d,momentum_20d,momentum_60d,rsi_14,price_to_sma_20d,price_to_sma_50d,volume_zscore_20d,excess_return_20d_vs_spy,intraday_range,beta_20d_spy
0,2014-01-02,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.009058,NaN
1,2014-01-03,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.024530,NaN
2,2014-01-06,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.024268,NaN
3,2014-01-07,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.014888,NaN
4,2014-01-08,AAPL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.012641,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78463,2025-12-24,TXN,0.015130,0.094950,0.016532,0.094950,-0.027317,42.843447,0.000438,0.047161,-1.596433,0.069173,0.006888,1.661551
78464,2025-12-26,TXN,0.003916,0.069731,0.016080,0.069731,-0.010706,34.882494,-0.004217,0.045292,-0.964098,0.051090,0.011816,1.574360
78465,2025-12-29,TXN,-0.003403,0.044096,0.015884,0.044096,-0.027764,35.663544,-0.012978,0.038043,-0.696523,0.034595,0.014514,1.532279
78466,2025-12-30,TXN,-0.019014,0.043173,0.015893,0.043173,-0.018491,38.053565,-0.016500,0.036399,-0.727059,0.030281,0.006955,1.581910


## 3. Add New Custom Features In The Notebook

This is where developers keep their freedom.

The toolkit does **not** force everyone to only use built-in features. A normal workflow is:

1. build a shared baseline feature set
2. add experimental features locally in the notebook
3. keep using the shared validation / portfolio / backtest layer afterward

Below we add a few simple handcrafted features:

- `mom_vol_ratio`
  A momentum-to-volatility ratio. This is a quick-and-dirty risk-adjusted trend signal.
- `trend_spread`
  The gap between short-term and medium-term trend distance.
- `quality_signal`
  A simple benchmark-relative momentum signal penalized by volatility.
- `range_volume_interaction`
  A rough interaction term between price range expansion and unusual volume.

In [19]:
frame = base_features.copy()

# Add a very small constant anywhere we divide so we do not create infinities.
eps = 1e-6

frame['mom_vol_ratio'] = frame['momentum_20d'] / (frame['vol_20d'].abs() + eps)
frame['trend_spread'] = frame['price_to_sma_20d'] - frame['price_to_sma_50d']
frame['quality_signal'] = frame['excess_return_20d_vs_spy'] - 0.5 * frame['vol_20d']
frame['range_volume_interaction'] = frame['intraday_range'] * frame['volume_zscore_20d']

custom_feature_names = [
    'mom_vol_ratio',
    'trend_spread',
    'quality_signal',
    'range_volume_interaction',
]

all_feature_names = base_feature_names + custom_feature_names
display(frame.loc[:, ['date', 'ticker'] + custom_feature_names].head())

,date,ticker,mom_vol_ratio,trend_spread,quality_signal,range_volume_interaction
0,2014-01-02,AAPL,NaN,NaN,NaN,NaN
1,2014-01-03,AAPL,NaN,NaN,NaN,NaN
2,2014-01-06,AAPL,NaN,NaN,NaN,NaN
3,2014-01-07,AAPL,NaN,NaN,NaN,NaN
4,2014-01-08,AAPL,NaN,NaN,NaN,NaN


## 4. Build Targets

We are going to fit three separate small MLPs with the exact same input features:

- one model for `expected_return`
- one model for `expected_alpha`
- one model for `expected_volatility`

This is not required, but it demonstrates the richer prediction contract supported by the toolkit.

Target builders used here:

- `make_forward_return_target(...)`
- `make_forward_alpha_target(...)`
- `make_forward_realized_vol_target(...)`

In [20]:
return_targets = make_forward_return_target(prices, horizon=horizon)
alpha_targets = make_forward_alpha_target(prices, horizon=horizon)
vol_targets = make_forward_realized_vol_target(prices, window=horizon)

target_frame = frame.merge(return_targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.merge(alpha_targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.merge(vol_targets, on=['date', 'ticker'], how='left')

# Drop rows only after all features and targets are assembled.
# This is the usual notebook pattern because long-window features and forward targets
# naturally create missing values near the beginning and end of each ticker history.
target_frame = target_frame.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

return_target_col = f'forward_return_{horizon}d'
alpha_target_col = f'forward_alpha_{horizon}d_vs_spy'
vol_target_col = f'forward_realized_vol_{horizon}d'

print('Modeling frame shape after dropping nulls:', target_frame.shape)
display(target_frame.head())

Modeling frame shape after dropping nulls: (76765, 21)


,date,ticker,return_5d,return_20d,vol_20d,momentum_20d,momentum_60d,rsi_14,price_to_sma_20d,price_to_sma_50d,...,excess_return_20d_vs_spy,intraday_range,beta_20d_spy,mom_vol_ratio,trend_spread,quality_signal,range_volume_interaction,forward_return_5d,forward_alpha_5d_vs_spy,forward_realized_vol_5d
0,2014-03-31,AAPL,-0.004544,0.017015,0.006837,0.017015,-0.023823,50.700744,0.006098,0.014112,...,0.001579,0.009092,0.415701,2.488167,-0.008015,-0.001840,-0.011534,-0.024723,-0.010446,0.146484
1,2014-04-01,AAPL,-0.006128,0.019595,0.006966,0.019595,0.007232,54.962582,0.014312,0.023227,...,0.011594,0.009416,0.468969,2.812689,-0.008915,0.008111,-0.005887,-0.033619,-0.016887,0.108590
2,2014-04-02,AAPL,0.005132,0.019142,0.006963,0.019142,0.003434,63.014392,0.015029,0.025053,...,0.008683,0.005935,0.465299,2.748703,-0.010024,0.005201,-0.005713,-0.022542,-0.013065,0.163988
3,2014-04-03,AAPL,0.002475,0.015149,0.007125,0.015149,0.003657,66.199534,0.007237,0.018312,...,0.008334,0.009020,0.501724,2.125846,-0.011076,0.004771,-0.011168,-0.028416,0.000583,0.172605
4,2014-04-04,AAPL,-0.009388,0.002602,0.007726,0.002602,-0.015561,55.243587,-0.005922,0.005939,...,0.008112,0.017713,0.631229,0.336674,-0.011861,0.004249,0.012090,-0.022959,0.003275,0.164261


## 5. Shared Train / Validation / Test Splits And Feature Scaling

This section does two very important things:

1. Uses the repo's shared split functions so the notebook respects the official date windows.
2. Standardizes features **using only the training split statistics**.

That second part matters a lot.

We do **not** want to normalize using future information from validation or test rows. So we compute mean and standard deviation from the train split only, then reuse those values everywhere else.

In [21]:
train = slice_split(target_frame, dataset_name, 'train', repo_root=repo_root)
val = slice_split(target_frame, dataset_name, 'val', repo_root=repo_root)
test = slice_split(target_frame, dataset_name, 'test', repo_root=repo_root)

print('Train rows:', len(train))
print('Val rows:', len(val))
print('Test rows:', len(test))

train_means = train[all_feature_names].mean()
train_stds = train[all_feature_names].std(ddof=0).replace(0.0, 1.0)

def standardize(feature_frame: pd.DataFrame) -> np.ndarray:
    standardized = (feature_frame[all_feature_names] - train_means) / train_stds
    return standardized.to_numpy(dtype=float)

X_train = standardize(train)
X_val = standardize(val)
X_test = standardize(test)

y_train_return = train[return_target_col].to_numpy(dtype=float)
y_val_return = val[return_target_col].to_numpy(dtype=float)
y_test_return = test[return_target_col].to_numpy(dtype=float)

y_train_alpha = train[alpha_target_col].to_numpy(dtype=float)
y_val_alpha = val[alpha_target_col].to_numpy(dtype=float)

y_train_vol = train[vol_target_col].to_numpy(dtype=float)
y_val_vol = val[vol_target_col].to_numpy(dtype=float)

print('X_train shape:', X_train.shape)
print('Feature count:', X_train.shape[1])

Train rows: 37695
Val rows: 13130
Test rows: 25940
X_train shape: (37695, 16)
Feature count: 16


## 6. Define A Very Basic MLP Class

This is a tiny PyTorch implementation of a feed-forward neural network for regression.

It is intentionally simple so the workflow is easy to understand:

- fully connected layers
- ReLU hidden activations
- linear output layer
- mean squared error loss
- mini-batch gradient descent
- validation loss tracking

This is **not** meant to be the most advanced PyTorch training setup. It is here so the notebook shows a real neural-network workflow while still staying readable.

In a real team workflow, this cell is exactly the part a researcher would replace with their own model implementation.

In [22]:
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


class TorchMLPRegressor(nn.Module):
    """A very small MLP regressor implemented in PyTorch.

    This keeps the same basic shape as the earlier NumPy example:
    - any number of hidden layers through `hidden_dims`
    - ReLU hidden activations
    - linear output layer for regression
    - mini-batch training with MSE loss

    Example:
    - hidden_dims=(32,) -> one hidden layer
    - hidden_dims=(64, 32) -> two hidden layers
    """

    def __init__(
        self,
        input_dim,
        hidden_dims=(64, 32),
        learning_rate=1e-3,
        epochs=40,
        batch_size=1024,
        random_state=42,
        device=None,
    ):
        super().__init__()

        self.input_dim = int(input_dim)
        self.hidden_dims = tuple(int(x) for x in hidden_dims)
        self.learning_rate = float(learning_rate)
        self.epochs = int(epochs)
        self.batch_size = int(batch_size)
        self.random_state = int(random_state)
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))

        # Keep training reproducible.
        random.seed(self.random_state)
        np.random.seed(self.random_state)
        torch.manual_seed(self.random_state)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.random_state)

        # Build a list like [input_dim, hidden_1, hidden_2, ..., 1]
        layer_dims = [self.input_dim, *self.hidden_dims, 1]
        layers = []

        for layer_idx, (in_dim, out_dim) in enumerate(zip(layer_dims[:-1], layer_dims[1:])):
            linear = nn.Linear(in_dim, out_dim)

            # Xavier initialization keeps early activations in a reasonable scale.
            nn.init.xavier_uniform_(linear.weight)
            nn.init.zeros_(linear.bias)

            layers.append(linear)

            # Final layer stays linear because this is a regression problem.
            if layer_idx < len(layer_dims) - 2:
                layers.append(nn.ReLU())

        self.network = nn.Sequential(*layers).to(self.device)
        self.loss_fn = nn.MSELoss()
        self.optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)

    def forward(self, X):
        return self.network(X)

    def predict(self, X):
        """Return predictions as a 1D NumPy array."""
        self.eval()
        X_tensor = torch.as_tensor(X, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            preds = self.forward(X_tensor).squeeze(-1)
        return preds.cpu().numpy()

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        """Train the model and return a pandas history DataFrame."""
        X_train = torch.as_tensor(X_train, dtype=torch.float32)
        y_train = torch.as_tensor(np.asarray(y_train).reshape(-1, 1), dtype=torch.float32)

        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
        )

        history = []

        for epoch in range(self.epochs):
            self.train()

            for X_batch, y_batch in train_loader:
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)

                self.optimizer.zero_grad()
                y_pred = self.forward(X_batch)
                loss = self.loss_fn(y_pred, y_batch)
                loss.backward()
                self.optimizer.step()

            train_pred = self.predict(X_train.numpy())
            train_loss = float(np.mean((train_pred - y_train.numpy().reshape(-1)) ** 2))

            row = {"epoch": epoch + 1, "train_loss": train_loss}

            if X_val is not None and y_val is not None:
                y_val_arr = np.asarray(y_val, dtype=float).reshape(-1)
                val_pred = self.predict(X_val)
                val_loss = float(np.mean((val_pred - y_val_arr) ** 2))
                row["val_loss"] = val_loss

            history.append(row)

        return pd.DataFrame(history)



## 7. Train Three Small MLPs

We are using the same feature matrix to train three related regressors:

- return model
- alpha model
- volatility model

This is a very common pattern in research projects:

- one common feature pipeline
- multiple prediction heads or target-specific models

The code below wraps the repeated steps in a helper function so the notebook stays readable.

In [23]:
device = 'cuda'
def train_single_target_model(target_name, y_train, y_val):
    model = TorchMLPRegressor(
        input_dim=X_train.shape[1],
        hidden_dims=(64, 32),
        learning_rate=1e-3,
        epochs=35,
        batch_size=1024,
        random_state=42,
    )
    history = model.fit(X_train, y_train, X_val=X_val, y_val=y_val)
    best_val_loss = float(history['val_loss'].min()) if 'val_loss' in history else float('nan')
    print(f'{target_name}: best validation loss = {best_val_loss:.8f}')
    return model, history

return_model, return_history = train_single_target_model('expected_return', y_train_return, y_val_return)
alpha_model, alpha_history = train_single_target_model('expected_alpha', y_train_alpha, y_val_alpha)
vol_model, vol_history = train_single_target_model('expected_volatility', y_train_vol, y_val_vol)

training_summary = pd.DataFrame(
    {
        'return_val_loss': return_history['val_loss'],
        'alpha_val_loss': alpha_history['val_loss'],
        'vol_val_loss': vol_history['val_loss'],
    }
)
display(training_summary.tail())

expected_return: best validation loss = 0.00383099
expected_alpha: best validation loss = 0.00250584
expected_volatility: best validation loss = 0.03773794


,return_val_loss,alpha_val_loss,vol_val_loss
30,0.003922,0.002564,0.040187
31,0.003909,0.002607,0.040599
32,0.003973,0.002612,0.040361
33,0.003890,0.002548,0.038568
34,0.003831,0.002506,0.038902


## 8. Create The Standardized Prediction Table

Now we convert raw model outputs into the shared prediction contract used by the rest of the toolkit.

Required columns:

- `date`
- `ticker`
- `horizon`
- `expected_return`

Optional columns we will also populate:

- `expected_alpha`
- `expected_volatility`
- `uncertainty`

For this notebook, `uncertainty` is just a simple constant based on validation RMSE for the return model. That is not a sophisticated uncertainty estimate. It is only here to demonstrate where that information would live in the shared schema.

In [24]:
test_return_pred = return_model.predict(X_test)
test_alpha_pred = alpha_model.predict(X_test)
test_vol_pred = np.clip(vol_model.predict(X_test), 1e-4, None)

# A toy notebook-level uncertainty estimate: use the square root of the best return-model val MSE.
# In a more serious project this could come from an ensemble, dropout approximation,
# quantile model, Bayesian model, or any custom confidence logic.
return_rmse = float(np.sqrt(return_history['val_loss'].min()))

predictions = test.loc[:, ['date', 'ticker']].copy()
predictions['horizon'] = horizon
predictions['expected_return'] = test_return_pred
predictions['expected_alpha'] = test_alpha_pred
predictions['expected_volatility'] = test_vol_pred
predictions['uncertainty'] = return_rmse

predictions = validate_prediction_frame(
    predictions,
    dataset_name=dataset_name,
    horizon=horizon,
    repo_root=repo_root,
)

display(predictions.head())
print('Prediction frame shape:', predictions.shape)

,date,ticker,horizon,expected_return,expected_alpha,expected_volatility,uncertainty
0,2022-01-03,AAPL,5,-0.013562,-0.011423,0.242078,0.061895
1,2022-01-03,ADBE,5,0.004547,-0.018660,0.314348,0.061895
2,2022-01-03,ADI,5,0.011327,0.002376,0.171097,0.061895
3,2022-01-03,AMAT,5,-0.002209,-0.005151,0.218810,0.061895
4,2022-01-03,AMD,5,-0.001723,0.005890,0.385267,0.061895


Prediction frame shape: (25940, 7)


## 9. Turn Predictions Into A Portfolio Object

The toolkit separates forecasting from portfolio construction.

Here we use the built-in `weights_from_predictions_risk_adjusted(...)` helper.

What it does:

- uses `expected_return / expected_volatility` as the score
- keeps the allocation long-only
- normalizes the scores so each row sums to `1.0`
- returns a `PortfolioWeights` object

This is a good default for demonstrations because it uses more of the prediction contract than a plain top-k rule.

In [25]:
portfolio = weights_from_predictions_risk_adjusted(
    predictions,
    dataset_name=dataset_name,
    strategy_name=model_name,
)

# The builder already validates internally, but doing it explicitly here makes the notebook
# clearer for new teammates.
validated_weights = validate_weights_frame(
    portfolio.weights,
    dataset_name=dataset_name,
    repo_root=repo_root,
)

print('Strategy name:', portfolio.strategy_name)
print('Weights frame shape:', validated_weights.shape)
display(validated_weights.head())

Strategy name: torch_mlp_forecast_example
Weights frame shape: (998, 26)


,AAPL,ADBE,ADI,AMAT,AMD,AMZN,AVGO,CRM,CSCO,GOOGL,...,MU,NFLX,NOW,NVDA,ORCL,PANW,QCOM,SPY,TSLA,TXN
date,,,,,,,,,,,,,,,,,,,,,
2022-01-03,0.000000,0.013312,0.060927,0.000000,0.000000,0.000000,0.000000,0.147308,0.092582,0.018249,...,0.000000,0.110664,0.025299,0.000000,0.0,0.078843,0.167126,0.075622,0.016007,0.012266
2022-01-04,0.000000,0.066298,0.000000,0.019729,0.050647,0.000000,0.001571,0.204416,0.009327,0.000000,...,0.012704,0.000000,0.181936,0.089223,0.0,0.000000,0.177185,0.000000,0.002540,0.000000
2022-01-05,0.000000,0.166200,0.000000,0.000000,0.128185,0.119411,0.083708,0.116554,0.000000,0.000000,...,0.013336,0.000000,0.068822,0.000000,0.0,0.095945,0.000000,0.000000,0.154498,0.000000
2022-01-06,0.015201,0.000000,0.163266,0.004695,0.050784,0.135946,0.000000,0.024719,0.132887,0.000000,...,0.000000,0.000000,0.057644,0.010523,0.0,0.135904,0.107960,0.000000,0.073447,0.071792
2022-01-07,0.066791,0.000000,0.000000,0.028677,0.084041,0.053360,0.000000,0.015845,0.085481,0.040588,...,0.089101,0.059784,0.046020,0.039777,0.0,0.005042,0.000000,0.000000,0.000000,0.000000


## 10. Run The Shared Backtest

This is where the toolkit gives you the most value as shared infrastructure.

The backtest layer will:

- load the shared dataset prices
- align rebalance dates to the next available trading day
- apply transaction costs from the dataset preset
- compare the strategy to benchmarks like `SPY`
- compute NAV, returns, turnover, and summary metrics

Because this is shared across the team, different notebooks remain comparable even if the model logic is very different.

In [26]:
result = backtest_weights(dataset_name, portfolio, repo_root=repo_root)
metrics = build_metrics(result)
artifact_paths = write_backtest_artifacts(result, output_dir)

metrics_table = pd.DataFrame(
    [{'metric': key, 'value': value} for key, value in sorted(metrics.items())]
).sort_values('metric').reset_index(drop=True)

display(metrics_table)
print('QuantStats report:', artifact_paths['quantstats_report'])

,metric,value
0,annual_return,-0.046082
1,annual_volatility,0.174181
2,average_turnover,0.506211
3,benchmark_total_return,0.507582
4,calmar,-0.076485
5,excess_return_vs_benchmark,-0.939335
6,max_drawdown,-0.602503
7,sharpe,-0.264567
8,sortino,-0.222772
9,total_return,-0.431753


QuantStats report: /content/Portfolio-Optimizer/runs/mlp_end_to_end_workflow/quantstats.html


## 11. Log The Run To MLflow

The toolkit keeps MLflow intentionally lightweight:

- local SQLite backend
- local artifact storage
- notebook-friendly logging helpers

The pattern here is the one you want teammates to reuse:

1. initialize MLflow locally
2. start a run with meaningful tags
3. log predictions, portfolio weights, and backtest results
4. let MLflow keep the artifact trail

In [28]:
!curl -fsSL https://tailscale.com/install.sh | sh

Installing Tailscale for ubuntu jammy, using method apt
+ mkdir -p --mode=0755 /usr/share/keyrings
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.noarmor.gpg
+ tee /usr/share/keyrings/tailscale-archive-keyring.gpg
+ chmod 0644 /usr/share/keyrings/tailscale-archive-keyring.gpg
+ curl -fsSL https://pkgs.tailscale.com/stable/ubuntu/jammy.tailscale-keyring.list
+ tee /etc/apt/sources.list.d/tailscale.list
# Tailscale packages for ubuntu jammy
deb [signed-by=/usr/share/keyrings/tailscale-archive-keyring.gpg] https://pkgs.tailscale.com/stable/ubuntu jammy main
+ chmod 0644 /etc/apt/sources.list.d/tailscale.list
+ apt-get update
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://pkgs.tailscale.com/stable/ubuntu jammy InRelease                 
Get:5 https://cli.github.

In [ ]:
# Colab-only: connect this runtime to your tailnet in userspace mode.
# Store TS_AUTHKEY in Colab Secrets, not inline in the notebook.
import os

!curl -fsSL https://tailscale.com/install.sh | sh
!nohup tailscaled \
    --tun=userspace-networking \
    --socks5-server=127.0.0.1:1055 \
    --outbound-http-proxy-listen=127.0.0.1:1055 \
    >/tmp/tailscaled.log 2>&1 &

!tailscale up --authkey="$TS_AUTHKEY" --hostname="colab-mlflow" --reset

os.environ["HTTP_PROXY"] = "http://127.0.0.1:1055"
os.environ["HTTPS_PROXY"] = "http://127.0.0.1:1055"
os.environ["NO_PROXY"] = "127.0.0.1,localhost"

print("Tailscale proxy configured for this Colab runtime.")

TimeoutException: Requesting secret TS_AUTHKEY timed out. Secrets can only be fetched when running from the Colab UI.

In [41]:
mlflow_layout = init_mlflow(repo_root)
print('MLflow tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name=model_name,
    dataset_name=dataset_name,
    tags={
        'workflow': 'mlp_end_to_end_workflow',
        'model_family': 'mlp',
        'prediction_horizon': str(horizon),
    },
    repo_root=repo_root,
):
    import mlflow

    mlflow.log_params(
        {
            'model_name': model_name,
            'dataset_name': dataset_name,
            'horizon': horizon,
            'feature_count': len(all_feature_names),
            'base_feature_list': ','.join(base_feature_names),
            'custom_feature_list': ','.join(custom_feature_names),
            'hidden_dims': '64,32',
            'learning_rate': 1e-3,
            'epochs': 35,
            'batch_size': 1024,
            'portfolio_builder': 'weights_from_predictions_risk_adjusted',
            'cost_bps': spec.cost_bps,
        }
    )

    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)

print('MLflow logging complete.')

MLflow tracking URI: http://100.113.71.47:5000


KeyboardInterrupt: 

## 12. Inspect Results

At this point the notebook has produced:

- validated predictions
- validated portfolio weights
- a shared backtest result
- standardized performance metrics
- a QuantStats HTML report
- an MLflow run with artifacts and metrics

That is the full intended research loop for this repo.

In [ ]:
print('Top-level metrics:')
for key, value in sorted(result.metrics.items()):
    print(f'  {key}: {value:.6f}')

print('\nArtifact paths:')
for key, value in artifact_paths.items():
    print(f'  {key}: {value}')

display(result.nav.tail().to_frame('nav'))
display(result.returns.tail().to_frame('returns'))
display(result.turnover.tail().to_frame('turnover'))

In [ ]:
# ---------------------------------------------------------------------
# Final validation cell.
# ---------------------------------------------------------------------
# These checks are intentionally simple. They are the kind of sanity checks
# you want at the end of a notebook before you trust the output.
# ---------------------------------------------------------------------

assert {'total_return', 'annual_return', 'sharpe', 'max_drawdown'}.issubset(result.metrics)
assert validated_weights.index.is_monotonic_increasing
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert Path(artifact_paths['quantstats_report']).exists()
assert {'date', 'ticker', 'horizon', 'expected_return'}.issubset(predictions.columns)

print('End-to-end MLP workflow validated successfully.')